<a href="https://colab.research.google.com/github/force23airr/Option_trading/blob/main/3_statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this session, we look at how to calculate various statistics of a data set.

Remember that, `pandas` data frame is like Excel spreadsheet. Just like the Excel spreadsheet is easy for handling data, so is `pandas` data frame in Python; just like we can convert almost every data set (unless it is insanely enormous) into Excel spreadsheet, we can convert almost every data set into `pandas` data frame (even if the data set is insanely enormous, and even if the data set is not in Excel format).

Of course, importing an Excel spreadsheet data set is an easy job, as we have learned previously. We start with an existing `pandas` data frame imported from an Excel spreadsheet.

First step, as always, we import all the necessary libraries.

In [ ]:
import numpy
import pandas
import scipy.stats

#### 1. Simple statistics

In [ ]:
dataset = pandas.read_excel('https://media.pearsoncmg.com/ph/bp/bp_stock_econometrics_4_cw/content/datapages/data/CPS96_15.xlsx')
dataset.head()

The simplest command is `dataset.describe()`. Once you have a data set in Python with a name (in our case `dataset`), the function `.describe()` returns the summary statistics for the data set. Note that, however, it generates the statistics for each column.

In [ ]:
dataset.describe()

Note that the above output itself is also a dataframe.

In [ ]:
type(dataset.describe())

If you want to export the above output to an Excel file for plotting or whatever, you can use `.toexcel()` command:

In [ ]:
result = dataset.describe() # give the above output data frame a name
result.to_excel('result.xlsx') # write the data frame into an Excel file

So where is this `result.xlsx` file? It is actually located on a temporary directory: if you click on the folder-like icon on the left (under the key icon), you will see the `result.xlsx` and you can download it to your local computer.

Now, going back to our original data set that has multiple columns (or variables), how do you obtain the summary statistics for each one?

Let's say I am interested in the summary statistics of `age` variable/column.

In [ ]:
dataset['age'].describe()

In [ ]:
print(dataset['age'].mean())
print(dataset['age'].median())
print(dataset['age'].var())
print(dataset['age'].std())
print(dataset['age'].min())
print(dataset['age'].max())
print(dataset['age'].quantile(0.25))
print(dataset['age'].quantile(0.5))
print(dataset['age'].quantile(0.75))

#### 2. Confidence interval and hypothesis testing

##### 2.1 Hypothesis testing: test population mean from one sample

We head straight to the core of testing whether a population mean equals to a hypothesized value or not.

\begin{align}
H_{0}: \ & \mu = a \, , \\
H_{1}: \ & \mu \neq a \, .
\end{align}

We obtain a sample data (with $n$ observations) from the population data:
$$
(y_{1}, y_{2}, \cdots, y_{n})
$$

Suppose the sample average is $\overline{Y}$, and the sample standard deviation is $s_Y$, then the t-statistic for testing the above hypothesis is
$$
t = \frac{\overline{Y}-a}{\frac{s_{Y}}{\sqrt{n}}} \, . \tag{1}
$$

In principle, since now we know how to use Python to calculate the mean and the standard deviation of a sample of data, calculating $(1)$ is easy by using Python (as a calculator, basically).



If we keep using the above dataset, and we want to test if the mean of the hourly wage for the population is $15$ or not, I can use

In [ ]:
t_statistic = (dataset['ahe'].mean()-15)/(dataset['ahe'].std()/numpy.sqrt(len(dataset)))
print(t_statistic)

The p-value for the calculated `t_statistic`, for the two-tailed test case, is
$$
2 \times \Pr (t > | \text{t_statistic} | )=2 \times \Pr (t > \text{t_statistic}) = 2 \times (1-\Pr (t < \text{t_statistic}))
$$
because in our example, the `t_statistic` is a positive number.

In [ ]:
p_value = 2*(1-scipy.stats.t.cdf(t_statistic, len(dataset)-1))
print(p_value)

But there is an even easier way of calculating $(1)$ in the `scipy` library, in one step, if we want to test the mean is equal to $1$ or not based on the sample data `df`:
\begin{align}
H_{0}: \ & \mu = 15 \, , \\
H_{1}: \ & \mu \neq 15 \, .
\end{align}
(i.e., you feed the data `df` directly without first calculating its mean or standard deviation)

In [ ]:
(t_statistic, p_value) = scipy.stats.ttest_1samp(dataset['ahe'], 15)
print(t_statistic)
print(p_value)

To see how the p-value is calculated (for the two-tail test), look at this:

In [ ]:
2*(1-scipy.stats.t.cdf(24.41378586, len(dataset)-1))

(There is a Z-statistic counterpart corresponding to (1), which is used when the population data variance $\sigma_Y$ is known:
$$
Z = \frac{\overline{Y}-a}{\sigma_{Y}/\sqrt{n}} \, . \tag{2}
$$
However, we will rarely be in the position when the population variance is known but its mean is unknown and is to be estimated.
Therefore, (2) is most likely of limited practical use.
When $\sigma_Y$ is unknown, and we have to use the sample variance $s_Y$, in most scenarios, t statistic (1) is the right way to go, and, although it converges to the standard normal distribution in the limit, the normal distribution is always an approximation. It is somewhat incongruous that, like in Stock and Watson (2019), you use the t statistic (1) yet treat it as a draw from a normal distribution for inference purposes.)

##### 2.2 Confidence interval: for population mean based on one sample

The textbook is very terse on the confidence interval related to the mean estimated from a sample of data. Here is the accurate and complete formula for the confidence interval based on the sample of data $(y_{1}, y_{2}, \cdots, y_{n})$:

$$
\overline{Y}- t_{\alpha /2 } \frac{s_Y}{\sqrt{n}} \le \mu \le \overline{Y}+ t_{\alpha /2 } \frac{s_Y}{\sqrt{n}} \, , \tag{2}
$$

where $t_{\alpha /2 }$ is the critical value for an upper-tail probability of $\alpha /2$ (i.e., a cumulative area of $1-\frac{\alpha}{2}$) from the $t$ distribution with $n-1$ degree of freedom.

Here, the confidence level is $(1-\alpha) \times 100$, if the significance level is $\alpha$.

Looking at the formula ($2$), we know how to calculate $\overline{Y}$ and $s_Y$, and we know how to calculate $t_{\alpha /2}$ (remember it is the critical value corresponding to the accumulative probability of $1-\frac{\alpha}{2}$), therefore, it is certainly within our reach to calculate out ($2$) based on these individual components.

In [ ]:
dataset['age'].mean()-scipy.stats.t.ppf(0.975, len(dataset)-1)*dataset['age'].std()/numpy.sqrt(len(dataset))
# the lower bound

In [ ]:
dataset['age'].mean()+scipy.stats.t.ppf(0.975, len(dataset)-1)*dataset['age'].std()/numpy.sqrt(len(dataset))
# the upper bound

In formula (2), $t_{\alpha /2 } \frac{s_Y}{\sqrt{n}}$ is called Margin of Error, measuring the degree of uncertainty due to sampling error. Therefore, we can calculate the margin of error as such:

In [ ]:
scipy.stats.t.ppf(0.975, len(dataset)-1)*dataset['age'].std()/numpy.sqrt(len(dataset))

Alternatively, you can calculate the margin of error as one half of the width of the confidence interval (see formula (2)).


If you want to be fancy, you can opt to use a function in `scipy.stats` to do the above in one step (again, you feed the data `dataset` into the formula):

In [ ]:
confidence_interval = scipy.stats.t.interval(0.95, len(dataset)-1, loc=dataset['age'].mean(), scale=scipy.stats.sem(dataset['age']))
print(confidence_interval)

where `sem` function from `scipy.stats` calculates the standard error of the sample mean ($\frac{s_Y}{\sqrt{n}}$).

##### 2.3 Hypothesis testing: test differences of means based on unmatched samples

Even though the textbook (Stock and Watson, 2019) warns about using the t distribution for testing the difference of two population means based on unmatched (or unpaired) samples. Yet the authors admit that however, in practice, when the sample size is above $15$, using the t distribution with pooled standard error formula rarely poses any problems. You have to be aware that the t distribution with pooled standard error is the standard procedure for testing the difference of two population means in most statistical software and will be the one we go to when testing two population means using two unmatched samples.

To fix the idea, suppose there is a sample $1$ drawn from a population data $1$, and a sample $2$ drawn from a population data $2$. The sample size, sample mean, and sample standard deviation for the sample $i$ ($i=1,2$) are $(n_{i}, \overline{X}_{i}, S_{i})$.

To test whether the difference of the population means $\mu_{1} - \mu_{2}$ is equal to $d$ or not (two-tailed test; you should be able to work out the one-tailed test version), the null and alternative hypotheses are:
\begin{align} \tag{3}
H_{0}: \ & \mu_{1}-\mu_{2} = d \, , \\
H_{1}: \ & \mu_{1}-\mu_{2} \neq d \, .
\end{align}


The t statistic associated with this hypothesis test is:
$$ \tag{4}
t = \frac{(\overline{X}_{1}-\overline{X}_{2})-d}{S_{p} \sqrt{\frac{1}{n_1}+\frac{1}{n_2}}} \, ,
$$
with the degree of freedom $n_{1}+n_{2}-2$, where $S_p$ is the pooled standard deviation from both samples calculated as follows:
$$
S_{p} = \frac{(n_{1}-1) S^2_{1}+ (n_{2}-1) S^2_{1}}{(n_{1}-1)+(n_{2}-1)} \, .
$$

(A special case is $d =0$, then we are testing whether the two population means are the same or not.)

Apparently, with the Sample $1$ and Sample $2$, we can compute $(n_{i}, \overline{X}_{i}, S_{i})$ $(i=1, 2)$, and use Python as a calculator to compute the t statistic above and then the associated p-value. Nothing wrong with that.

But, we can also directly feed the data into the Python formula `scipy.stats.ttest_ind()` for the test without first calculating the means, standard deviations, etc.

Suppose we have two samples: `var1` and `var2`([simple2samples.xlsx](https://docs.google.com/spreadsheets/d/17XmDNZDQqWWmivc4uHQVl7eTqv0iktLy/edit?usp=drive_link&ouid=110351823593825164156&rtpof=true&sd=true), if you want to download):

In [ ]:
from google.colab import files
import pandas

uploaded = files.upload()
file_path = list(uploaded.keys())[0]
dataset = pandas.read_excel(file_path)
#upload simplesample2.xlsx to read in here.

dataset.head()

In [ ]:
t_stat, p_value = scipy.stats.ttest_ind(dataset['var1'], dataset['var2'])
print(f"T-statistic: {t_stat}, P-value: {p_value}")

Or, if we know the means and standard devations of the two samples, we can use another formula:

In [ ]:
print(dataset['var1'].mean())
print(dataset['var1'].std())
print(dataset['var1'].count())
print(dataset['var2'].mean())
print(dataset['var2'].std())
print(dataset['var2'].count())

In [ ]:
t_stat, p_value = scipy.stats.ttest_ind_from_stats(mean1=dataset['var1'].mean(), std1=dataset['var1'].std(), nobs1=dataset['var1'].count(),
                                                   mean2=dataset['var2'].mean(), std2=dataset['var2'].std(), nobs2=dataset['var2'].count())
print(f"t-statistic: {t_stat}, P-value: {p_value}")

(If you want to estimate the confidence interval based on (4), you can still use `scipy.stats.t.interval` function with `loc=` and `scale=` inputs computed correspondingly by the components in (4). Think about how you can do it.)

##### 2.4 Hypothesis testing: test differences of means based on matched samples

We know the best data for testing the difference between two population means are the samples from both populations, and the sample observations are matched one-to-one by some characteristics. For example, if comparing two companies' sales performance, you would like to have sales figures for both companies in year 1, sales figures for both companies in year 2, etc. Then we can legitimately compare the differences in these sales figures for both companies, because each pair is from the same year.

We can pretend our [simple2samples.xlsx](https://docs.google.com/spreadsheets/d/17XmDNZDQqWWmivc4uHQVl7eTqv0iktLy/edit?usp=drive_link&ouid=110351823593825164156&rtpof=true&sd=true) data are sales figures for Company 1 (`var1`) and for Company 2 (`var2`) in five select years. Each row consists of sales figures data for a particular year. This will be two matched samples.

Suppose we want to test whether the average sales are the same for both companies or not. Since the data are matched, we can just take the difference between these two samples and treat the resulting difference column as the single data sample to test. You can do this in Excel, but if you want to do it here in Python, it's simple:

In [ ]:
dataset['diff']=dataset['var1']-dataset['var2']

Then, back to our one-sample formula in coding (testing the difference is zero or not):

In [ ]:
(t_statistic, p_value) = scipy.stats.ttest_1samp(dataset['diff'], 0)
print(t_statistic)
print(p_value)